In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
)
from imblearn.over_sampling import SMOTE


In [2]:
# Carregando os dados
data = pd.read_csv('dados.csv')

# Visualizando as primeiras linhas
data.head()

# Verificando colunas categóricas
categorical_columns = ['Gender', 'State (Location)', 'Performance']

plt.figure(figsize=(18, 6))
for i, col in enumerate(categorical_columns, 1):
    plt.subplot(1, len(categorical_columns), i)
    sns.countplot(data[col])
    plt.title(f'Distribuição de {col}')
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'dados.csv'

In [ ]:
# Removendo valores 'MD' na coluna 'Performance'
data = data[data['Performance'] != 'MD']

# Tratando valores 'MD' nas colunas específicas
for col in ['Analytical Skills 1', 'Domain Skills 1', 'Quantitative Ability 1']:
    data[col] = data[col].replace('MD', np.nan)
    data[col] = pd.to_numeric(data[col], errors='coerce')
    data[col].fillna(data[col].median(), inplace=True)


In [ ]:
# Aplicando regra do IQR para detectar e remover outliers
num_columns = data.select_dtypes(include=np.number).columns
q1 = data[num_columns].quantile(0.25)
q3 = data[num_columns].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

# Removendo outliers
data = data[~((data[num_columns] < limite_inferior) | (data[num_columns] > limite_superior)).any(axis=1)]


In [ ]:
# Removendo colunas irrelevantes
data = data.drop(['Candidate ID', 'Name', 'Month of Birth', 'Year of Birth', 'Gender', 'State (Location)'], axis=1)

# Separando variáveis preditoras e alvo
X, y = data.drop('Performance', axis=1), data['Performance']

# Codificação com LabelEncoder para variáveis categóricas
label_encoder = LabelEncoder()
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = label_encoder.fit_transform(X[col])

# Codificação do alvo
y = label_encoder.fit_transform(y)


In [ ]:
# Agrupando colunas relacionadas
X['Domain Skills'] = X[['Domain Skills 1', 'Domain Skills 2', 'Domain Test 3', 'Domain Test 4']].mean(axis=1)
X['Analytical Skills'] = X[['Analytical Skills 1', 'Analytical Skills 2', 'Analytical Skills 3']].mean(axis=1)
X['Quantitative Ability'] = X[['Quantitative Ability 1', 'Quantitative Ability 2', 'Quantitative Ability 3', 'Quantitative Ability 4']].mean(axis=1)
X['English'] = X[[' English 1', 'English 2', 'English 3', 'English 4']].mean(axis=1)

# Removendo colunas usadas no agrupamento
X = X.drop([
    'Domain Skills 1', 'Domain Skills 2', 'Domain Test 3', 'Domain Test 4',
    'Analytical Skills 1', 'Analytical Skills 2', 'Analytical Skills 3',
    'Quantitative Ability 1', 'Quantitative Ability 2', 'Quantitative Ability 3', 'Quantitative Ability 4',
    ' English 1', 'English 2', 'English 3', 'English 4'
], axis=1)


In [ ]:
# Seleção de características mais relevantes
selecao = SelectKBest(score_func=f_classif, k=5)
X_novo = selecao.fit_transform(X, y)
colunas_selecionadas = X.columns[selecao.get_support()]
print(f'Colunas selecionadas: {colunas_selecionadas}')


In [ ]:
# Dividindo os dados em treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X_novo, y, test_size=0.2, random_state=42)

# Aplicando SMOTE para balancear as classes
smote = SMOTE(random_state=42)
X_treino_bal, y_treino_bal = smote.fit_resample(X_treino, y_treino)

In [ ]:
# Ajuste de hiperparâmetros para RandomForest
rf = RandomForestClassifier(random_state=42)
param_grid_rf = {
    'n_estimators': [100, 300, 500],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}
grid_search_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='accuracy')
grid_search_rf.fit(X_treino_bal, y_treino_bal)

# Melhor modelo de RandomForest
best_rf = grid_search_rf.best_estimator_
previsoesRF = best_rf.predict(X_teste)

# Métricas
print("RandomForest")
print(f"Acurácia: {accuracy_score(y_teste, previsoesRF):.4f}")
print(f"F1 Score: {f1_score(y_teste, previsoesRF, average='weighted'):.4f}")
print(f"Precision: {precision_score(y_teste, previsoesRF, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_teste, previsoesRF, average='weighted'):.4f}")


In [ ]:
# Ajuste de hiperparâmetros para GradientBoosting
gbc = GradientBoostingClassifier(random_state=42)
param_grid_gbc = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
}
grid_search_gbc = GridSearchCV(gbc, param_grid_gbc, cv=5, scoring='accuracy')
grid_search_gbc.fit(X_treino_bal, y_treino_bal)

# Melhor modelo de GradientBoosting
best_gbc = grid_search_gbc.best_estimator_
previsoesGBC = best_gbc.predict(X_teste)

# Métricas
print("GradientBoosting")
print(f"Acurácia: {accuracy_score(y_teste, previsoesGBC):.4f}")
print(f"F1 Score: {f1_score(y_teste, previsoesGBC, average='weighted'):.4f}")
print(f"Precision: {precision_score(y_teste, previsoesGBC, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_teste, previsoesGBC, average='weighted'):.4f}")


In [ ]:
# Exibindo resultados lado a lado
print("Resultados Comparativos")
print(f"Acurácia (RandomForest): {accuracy_score(y_teste, previsoesRF):.4f}")
print(f"Acurácia (GradientBoosting): {accuracy_score(y_teste, previsoesGBC):.4f}")
